# 6.1 - Diffusion Models

**Module 6 - Image & Multimodal Generation** | Netsetos GenAI Engineering

This notebook works through Diffusion Models hands-on: The noise mixer: one function is the whole forward process; The forward process: clean to static in six snapshots; The noise schedule: linear vs cosine; Train the denoiser: predict the noise; Sampling: reverse from pure noise.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

This cell is just environment prep - a commented-out `pip install` for PyTorch and a note that the lesson uses seeded random values for reproducibility. Nothing runs and nothing is defined yet; the real work starts in the next cell.

In [ ]:
# Install dependencies (if running in Colab or fresh environment)
# Uncomment the next line if needed:
# !pip install torch -q

# Reproducibility - the lesson uses seeded random values throughout

**Explanation**

A placeholder setup cell. It carries no live code - only a commented install line you uncomment if PyTorch is missing, plus a reminder that the lesson seeds its randomness so results are repeatable.

**How the code works - step by step**
- **Commented `pip install torch`** - uncomment it only in Colab or a fresh environment where PyTorch is not already present.
- **Reproducibility note** - flags that the lesson works with seeded random values so the same code gives the same noise every run.

**In one line:** environment prep, no model and no output.

**What you'll see:** (no output - environment prep)

## 1 - The noise mixer: one function is the whole forward process

Before any network, we need something to noise and a way to noise it. `make_sample_image` gives us a toy 64x64 image (a white square on black) so the math stays visible, and `add_noise` blends that image with Gaussian noise by a single ratio. That one blend is the entire forward process of diffusion - understand it and the rest of the lesson is detail.

In [ ]:
import torch

def make_sample_image(size: int = 64) -> torch.Tensor:
    """A toy 'image': a white square on black. A real one would load from disk."""
    img = torch.zeros(1, size, size)   # 1 channel, 64x64, all black
    img[0, 16:48, 16:48] = 1.0      # a white square in the middle
    return img

def add_noise(image: torch.Tensor, alpha_bar: torch.Tensor):
    """Mix image and noise. alpha_bar in [0,1] is how much IMAGE survives."""
    noise = torch.randn_like(image)                  # Gaussian noise, same shape
    noisy = torch.sqrt(alpha_bar) * image + torch.sqrt(1 - alpha_bar) * noise
    return noisy, noise                              # return the noise too - it is the 'answer'

image = make_sample_image()
noisy, noise = add_noise(image, torch.tensor(0.5))
print(image.shape, noisy.shape)
# Output: torch.Size([1, 64, 64]) torch.Size([1, 64, 64])
print(f"alpha_bar=0.5 -> half image, half noise")
# Output: alpha_bar=0.5 -> half image, half noise

**Explanation**

This cell defines the two primitives reused all lesson: a stand-in image and a noise mixer. The key move is in `add_noise` - it combines a fraction of the clean image with a fraction of fresh Gaussian noise, and crucially it also returns the noise it added, because that noise is the label the network will later be trained to predict.

**How the code works - step by step**
- **`make_sample_image`** - builds a 1-channel 64x64 tensor of zeros (black) and sets a central 32x32 block to 1.0 (white), a visible stand-in for a real photo.
- **`add_noise`** - draws Gaussian noise the same shape as the image, then returns `sqrt(alpha_bar) * image + sqrt(1 - alpha_bar) * noise`. The coefficients are square roots because their squares are variances that sum to 1, so total variance stays fixed at every step (nothing blows up or fades).
- **Returns the noise too** - the second return value is the exact noise added; that is the training target in cell 4.
- **`alpha_bar`** - the single knob: 1.0 keeps the image clean, 0.0 gives pure noise.

**In one line:** mix `alpha_bar` of image with `1 - alpha_bar` of noise, and hand back both the noisy image and the noise.

**What you'll see:** Two prints confirming the shapes match (`torch.Size([1, 64, 64])` for both image and noisy), and a line noting that `alpha_bar=0.5` means half image, half noise.

## 2 - The forward process: clean to static in six snapshots

Now we run `add_noise` across a descending ladder of `alpha_bar` values to watch the same image dissolve. This is the forward process made concrete - and the point is that it involves no learning at all, only fixed arithmetic. The end state is where generation will later begin.

In [ ]:
# THE FORWARD PROCESS: walk an image from clean to pure noise.
# x_t = sqrt(alpha_bar_t) * x_0 + sqrt(1 - alpha_bar_t) * noise

alpha_bars = [0.99, 0.80, 0.50, 0.20, 0.05, 0.01]
for ab in alpha_bars:
    noisy, _ = add_noise(image, torch.tensor(ab))
    pct = round(100 * (1 - ab))
    print(f"alpha_bar={ab:.2f}  ->  ~{pct:>2d}% noise")

# Output: alpha_bar=0.99  ->  ~ 1% noise   (square clearly visible)
# Output: alpha_bar=0.80  ->  ~20% noise   (square slightly grainy)
# Output: alpha_bar=0.50  ->  ~50% noise   (square fading)
# Output: alpha_bar=0.20  ->  ~80% noise   (barely there)
# Output: alpha_bar=0.05  ->  ~95% noise   (nearly gone)
# Output: alpha_bar=0.01  ->  ~99% noise   (pure static)

**Explanation**

A pure demonstration cell, not a model call. It sweeps `alpha_bar` from 0.99 down to 0.01 and reports the corresponding noise percentage, showing the image walk from crisp to pure static. There is no network here - the forward direction is always just arithmetic; only the reverse direction needs to be learned.

**How the code works - step by step**
- **`alpha_bars` list** - six survival ratios from 0.99 (almost clean) to 0.01 (almost pure noise).
- **Loop** - calls `add_noise` at each ratio and computes `pct = round(100 * (1 - ab))`, the variance share that is now noise.
- **No network, no gradients** - this is fixed math, underlining that the hard part is only ever the reverse process.
- **End state** - at `alpha_bar=0.01` the result is essentially indistinguishable from `torch.randn`, which is exactly where sampling starts in cell 5.

**In one line:** as `alpha_bar` falls, the square fades to static - the forward process in six frames.

**What you'll see:** Six lines mapping each `alpha_bar` to a noise percentage, from `alpha_bar=0.99 -> ~1% noise` (square clearly visible) down to `alpha_bar=0.01 -> ~99% noise` (pure static).

## 3 - The noise schedule: linear vs cosine

The schedule decides *how fast* signal dies across the timesteps, and that choice measurably affects image quality. Here we implement both the original DDPM linear schedule and the improved cosine schedule, then convert each to the cumulative `alpha_bar` curve and compare them at two timesteps to see why cosine wins.

In [ ]:
def linear_schedule(T=1000, beta_start=1e-4, beta_end=0.02):
    """The original DDPM schedule: betas ramp up linearly."""
    return torch.linspace(beta_start, beta_end, T)

def cosine_schedule(T=1000):
    """Improved DDPM: signal fades on a gentler cosine curve."""
    steps = torch.arange(T + 1) / T
    abar = torch.cos((steps + 0.008) / 1.008 * torch.pi / 2) ** 2
    abar = abar / abar[0]
    betas = 1 - (abar[1:] / abar[:-1])
    return betas.clamp(max=0.999)

def alpha_bars_from(betas):
    return torch.cumprod(1.0 - betas, dim=0)   # cumulative product

T = 1000
lin = alpha_bars_from(linear_schedule(T))
cos = alpha_bars_from(cosine_schedule(T))
print(f"step  50: linear abar={lin[50]:.3f}  cosine abar={cos[50]:.3f}")
print(f"step 500: linear abar={lin[500]:.3f}  cosine abar={cos[500]:.3f}")
# Output: step  50: linear abar=0.970  cosine abar=0.992
# Output: step 500: linear abar=0.078  cosine abar=0.492

**Explanation**

This cell defines the two standard schedules and the converter that turns per-step betas into the cumulative `alpha_bar` the mixer needs. The comparison is the payoff: at the midpoint of the process the linear schedule has already crushed the signal to a sliver while cosine still retains about half, which is why cosine keeps more timesteps learnable.

**How the code works - step by step**
- **`linear_schedule`** - the original DDPM choice: betas ramp linearly from `1e-4` to `0.02` over T steps via `torch.linspace`.
- **`cosine_schedule`** - the improved DDPM choice: builds the `alpha_bar` curve directly from a shifted cosine so signal fades more gently, then derives betas and clamps them below 0.999.
- **`alpha_bars_from`** - turns per-step betas into cumulative survival with a single `torch.cumprod(1 - betas)`.
- **Comparison** - prints both schedules' `alpha_bar` at steps 50 and 500 to expose the gap.

**In one line:** cosine keeps more of the image alive through the middle timesteps, so more steps carry learnable signal.

**What you'll see:** Two lines: at step 50 the schedules are close (linear 0.970 vs cosine 0.992), but at step 500 they diverge sharply - linear has collapsed to 0.078 while cosine still holds 0.492.

## 4 - Train the denoiser: predict the noise

This is the only learned component and the heart of the lesson. We define a small CNN that takes a noisy image plus its timestep and predicts the noise inside it, then train it with a plain MSE loss. The trick worth internalizing: the network predicts the *noise that was added*, not the clean image - an easier target that makes training stable.

In [ ]:
import torch.nn as nn

class SimpleDenoiser(nn.Module):
    """Predicts the noise in a noisy image. A real model uses a U-Net or a DiT;
    this CNN captures the idea in a few layers."""
    def __init__(self, ch=1, hidden=64, T=1000):
        super().__init__()
        self.time_embed = nn.Embedding(T, hidden)        # timestep -> vector (lookup table)
        # two NAMED blocks so we can inject the timestep cleanly between them
        self.in_conv = nn.Sequential(nn.Conv2d(ch, hidden, 3, padding=1), nn.ReLU())
        self.out_net = nn.Sequential(
            nn.Conv2d(hidden, hidden, 3, padding=1), nn.ReLU(),
            nn.Conv2d(hidden, ch, 3, padding=1),         # out: predicted noise
        )

    def forward(self, x, t):
        emb = self.time_embed(t).unsqueeze(-1).unsqueeze(-1)  # (B,hidden,1,1)
        h = self.in_conv(x) + emb        # add the timestep vector to every pixel (broadcast)
        return self.out_net(h)           # run the rest of the network

model = SimpleDenoiser(T=T)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
abar = alpha_bars_from(linear_schedule(T))

for step in range(300):
    x0 = torch.stack([make_sample_image() for _ in range(16)])   # (16,1,64,64)
    t = torch.randint(0, T, (16,))                            # a random timestep per image
    ab = abar[t].view(-1, 1, 1, 1)
    noise = torch.randn_like(x0)
    x_t = torch.sqrt(ab) * x0 + torch.sqrt(1 - ab) * noise   # forward process

    pred = model(x_t, t)                                   # predict the noise
    loss = nn.functional.mse_loss(pred, noise)             # compare to the REAL noise
    opt.zero_grad(); loss.backward(); opt.step()
    if (step + 1) % 100 == 0:
        print(f"step {step+1:3d}  loss {loss.item():.4f}")

# Output: step 100  loss 0.4117
# Output: step 200  loss 0.2098
# Output: step 300  loss 0.1413

**Explanation**

The whole cell is the complete DDPM training loop. Each iteration takes clean images, runs the forward process to noise them at random timesteps, asks the network what noise it sees, and minimizes the mean-squared error against the noise we actually added. The timestep embedding lets one network cover every noise level, and there is no adversary or exotic loss - just MSE.

**How the code works - step by step**
- **`SimpleDenoiser.__init__`** - defines a timestep embedding (`nn.Embedding`, a learned lookup vector per step), an input conv block, and an output block whose last conv produces the predicted noise.
- **`forward`** - looks up the timestep vector and adds it (broadcast) to the first conv's output, so every pixel 'knows' the noise level, then runs the rest of the net.
- **Training loop** - each step stacks 16 clean images, picks a random timestep per image, runs the forward process to make `x_t`, and keeps the exact `noise`.
- **Loss** - `mse_loss(pred, noise)` compares the prediction to the real noise; `zero_grad -> backward -> step` is the standard PyTorch weight update that makes the loss fall.

**In one line:** noise a clean image, ask the net what noise it sees, and minimize MSE against the truth - that is all of DDPM training.

**What you'll see:** Three progress lines every 100 steps showing the loss dropping steadily - `step 100 loss 0.4117`, `step 200 loss 0.2098`, `step 300 loss 0.1413` - as the network gets better at spotting noise at every level.

## 5 - Sampling: reverse from pure noise

With a trained denoiser, generation is the reverse process: start from pure static and subtract a slice of predicted noise over and over until an image emerges. This cell implements one full DDPM sampling loop, from `t=T-1` down to `t=0`, and shows why the many network calls are the latency cost of diffusion.

In [ ]:
@torch.no_grad()
def sample(model, T=1000, size=64):
    """Generate one image by denoising pure noise from t=T-1 down to t=0."""
    betas = linear_schedule(T)
    alphas = 1.0 - betas
    abar = torch.cumprod(alphas, dim=0)

    x = torch.randn(1, 1, size, size)             # START: pure noise
    for t in reversed(range(T)):
        pred_noise = model(x, torch.tensor([t]))    # what noise is in x?
        a, ab, b = alphas[t], abar[t], betas[t]
        z = torch.randn_like(x) if t > 0 else 0  # no fresh noise on last step
        # one DDPM reverse step: remove a slice of predicted noise
        x = (1 / torch.sqrt(a)) * (x - (b / torch.sqrt(1 - ab)) * pred_noise) \
            + torch.sqrt(b) * z
    return x.clamp(0, 1)

img = sample(model, T=T)
print(img.shape, f"range [{img.min():.2f}, {img.max():.2f}]")
# Output: torch.Size([1, 1, 64, 64]) range [0.00, 1.00]
# A square emerges from noise - we trained on squares, so it generates squares.

**Explanation**

This is the inference counterpart to cell 4 - a generation loop, not training. It begins at `torch.randn` (the same static the forward process ended on) and applies the trained model in a loop, each iteration removing a beta-scaled slice of predicted noise and re-injecting a touch of fresh randomness to stay stochastic, except on the final clean step. Running it hundreds of times is what turns noise into a recognizable square.

**How the code works - step by step**
- **`@torch.no_grad()`** - sampling needs no gradients, so autograd is switched off for speed.
- **Start** - `x = torch.randn(1, 1, size, size)`, pure noise identical to the forward process's end state.
- **Reverse loop** - for each `t` from T-1 down to 0: predict the noise, apply the DDPM update `(1/sqrt(a)) * (x - (b/sqrt(1-ab)) * pred_noise)`, and add fresh noise `z` except on the last step.
- **`x.clamp(0, 1)`** - squeezes the final image back into valid pixel range.
- **Cost note** - the loop is hundreds to a thousand network calls; DDIM rewrites this as a deterministic step you can take in ~20-50 jumps instead.

**In one line:** start from static, subtract a predicted slice of noise each step, and a trained-on square emerges.

**What you'll see:** One line reporting the generated tensor shape and value range - `torch.Size([1, 1, 64, 64]) range [0.00, 1.00]` - with a comment that a square emerges from noise because the model was trained on squares.

You built diffusion end to end from six short cells: `add_noise` is the entire forward process, the schedule cells set how fast signal fades, the training loop teaches one CNN to predict noise with plain MSE, and the sampling loop reverses that process from pure static into an image. Every production system - Stable Diffusion, Flux, SD3.5 - is this same loop with a heavier backbone (U-Net or Diffusion Transformer), a VAE latent space, and a text encoder wired in through cross-attention. Next in Module 6: Lesson 6.2 runs real checkpoints through the `diffusers` library and fine-tunes them with LoRA, and Lesson 6.4 opens up the CLIP text-image embedding space the text encoder here relies on.